# Predicting next-month arrears
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on billing month, choose by mean CV ROC AUC, score holdout once).

**Feature decision.** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. alongside/after the outcome (87% arrears when 1 vs 1.4% when 0). It is not known at prediction time and is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='42accdc3f8c6ae3514896d73516d98c54e4a27c5422d4fe88e5c4fe1b00a14f7'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='903bb064736772428ae27c90c6c03ac7f9c28c7eb54a50c70b7601b731258ba6'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==20939 and len(hold_idx)==7536
is_hold=df.index.isin(hold_idx)
train=df[~is_hold]; hold=df[is_hold]
TARGET='arrears_next_month'
CAT=['tariff']; NUM=['occupants','kwh','bill_eur','late_payments_12m']
FEATURES=CAT+NUM
print(len(train),len(hold), train.billing_month.max(), hold.billing_month.min())

## Candidates (named before any scoring)

In [ ]:
def pre():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
CANDIDATES={
 'baseline_prior': Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',pre()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',pre()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',pre()),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'gbm_depth3': Pipeline([('p',pre()),('m',GradientBoostingClassifier(max_depth=3,n_estimators=150,learning_rate=0.05,random_state=0))]),
}
list(CANDIDATES)

## Walk-forward CV on training rows (by billing month; preprocessing is inside each fold's pipeline)

In [ ]:
months=np.sort(train.billing_month.unique())
N_SPLITS=5
# months split into N_SPLITS+1 contiguous blocks; fold k trains on blocks <=k, validates on block k+1
blocks=np.array_split(months,N_SPLITS+1)
folds=[]
for k in range(N_SPLITS):
    tr_m=np.concatenate(blocks[:k+1]); va_m=blocks[k+1]
    tr=np.where(train.billing_month.isin(tr_m))[0]; va=np.where(train.billing_month.isin(va_m))[0]
    folds.append((tr,va))
    print(k,len(tr),len(va),va_m[0],va_m[-1])
cv_scores={}
for name,est in CANDIDATES.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(train.iloc[tr][FEATURES],train.iloc[tr][TARGET])
        s.append(float(roc_auc_score(train.iloc[va][TARGET],m.predict_proba(train.iloc[va][FEATURES])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen=cv['mean'].idxmax(); print('chosen:',chosen)

## Refit on all training rows; score holdout once

In [ ]:
final=clone(CANDIDATES[chosen]).fit(train[FEATURES],train[TARGET])
holdout_auc=float(roc_auc_score(hold[TARGET],final.predict_proba(hold[FEATURES])[:,1]))
print('Holdout ROC AUC:',round(holdout_auc,4))

In [ ]:
pos=lambda idx:[int(i) for i in idx]
evidence=dict(
 features=FEATURES,
 candidates=list(CANDIDATES),
 cv_folds=[[pos(train.index[tr]),pos(train.index[va])] for tr,va in folds],
 cv_scores=cv_scores,
 chosen=chosen,
 holdout_rows=pos(hold.index),
 holdout_scores={chosen:holdout_auc},
)
json.dump(evidence,open('evidence.json','w'))